1. Importing Necessary Libraries

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import Input, Model, Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, GlobalAveragePooling2D, Dropout, concatenate, Bidirectional, LSTM
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix

2. Configuration

In [2]:
DATA_PATH = r''  
CSV_FILENAME = 'fruit_quality_train_detector_annotations.csv' 
IMG_SIZE = 100
NUM_CHANNELS = 3  
NUM_CLASSES = 2    
BATCH_SIZE = 64
SEED = 42
NUM_EPOCHS = 30
np.random.seed(SEED)
tf.random.set_seed(SEED)

3. Load Annotations CSV

In [3]:
csv_path = CSV_FILENAME
if DATA_PATH:
    candidate = os.path.join(DATA_PATH, CSV_FILENAME)
    if os.path.exists(candidate):
        csv_path = candidate

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Annotations CSV not found at: {csv_path}. Place the file or update CSV_FILENAME/DATA_PATH.")

df = pd.read_csv(csv_path, sep=None, engine='python')
print('Loaded CSV:', csv_path)
print('Rows:', len(df))
print('Columns:', df.columns.tolist())
df.head()

Loaded CSV: fruit_quality_train_detector_annotations.csv
Rows: 58578
Columns: ['image_id', 'label', 'isNight', 'split', 'adv_label']


,image_id,label,isNight,split,adv_label
0,E:\Fruit_Quality_Split\train\bad\IMG_2510.JPG,1,1,train,0
1,E:\Fruit_Quality_Split\train\good\IMG202007281...,0,1,train,0
2,E:\Fruit_Quality_Split\train\good\IMG_9493.JPG,0,1,train,0
3,E:\Fruit_Quality_Split\train\good\20190820_153...,0,1,train,0
4,E:\Fruit_Quality_Split\train\good\IMG202007281...,0,1,train,0


3. Build X/y for binary detection

In [4]:
if 'adv_label' not in df.columns:
    if 'source' in df.columns:
        df['adv_label'] = df['source'].apply(lambda s: 0 if str(s).strip().lower() == 'train_detector_annotations' else 1)
        print('Derived adv_label from source column.')
    elif 'image_id' in df.columns:
        df['adv_label'] = df['image_id'].apply(lambda p: 1 if 'adv_outputs' in str(p) else 0)
        print('Heuristically derived adv_label from image_id paths.')
    else:
        raise RuntimeError('No adv_label, source, or image_id column found to infer adversarial labels.')

if 'image_path' in df.columns:
    image_col = 'image_path'
elif 'image_id' in df.columns:
    image_col = 'image_id'
else:
    image_col = df.columns[0]
print('Using image column:', image_col)

def resolve_image_path(p):
    if not isinstance(p, str):
        return p
    p_norm = os.path.normpath(p)
    if os.path.isabs(p_norm) and os.path.exists(p_norm):
        return p_norm
    if os.path.exists(p_norm):
        return p_norm
    if DATA_PATH:
        cand = os.path.join(DATA_PATH, p_norm)
        if os.path.exists(cand):
            return cand
        cand2 = os.path.join(DATA_PATH, os.path.basename(p_norm))
        if os.path.exists(cand2):
            return cand2
    return p_norm

def build_Xy_binary(df_part, img_size=IMG_SIZE, num_channels=NUM_CHANNELS):
    X_images, X_isNight, y = [], [], []
    missing_count = 0
    unreadable_count = 0
    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        raw = str(row[image_col])
        img_path = raw.replace('\\','/')
        img_path = resolve_image_path(img_path)
        adv_lbl = int(row['adv_label'])
        is_night = int(row.get('isNight', 0))
        if not os.path.exists(img_path):
            missing_count += 1
            if missing_count <= 5:
                print('Missing image:', img_path)
            continue
        img = cv2.imread(img_path)
        if img is None:
            unreadable_count += 1
            if unreadable_count <= 5:
                print('Could not read image:', img_path)
            continue
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        if img.shape[2] == 4 and num_channels == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.resize(img, (img_size, img_size)).astype('float32') / 255.0
        X_images.append(img)
        X_isNight.append(is_night)
        y.append(adv_lbl)
    if len(X_images) == 0:
        raise RuntimeError('No images loaded — verify paths and CSV.')
    X_images = np.array(X_images)
    X_isNight = np.array(X_isNight).reshape(-1,1)
    y = np.array(y).astype('int32')
    return X_images, X_isNight, y

X_train_img, X_train_meta, y_train = build_Xy_binary(df[df['split']=='train'])
X_val_img,   X_val_meta,   y_val   = build_Xy_binary(df[df['split']=='valid'])
X_test_img,  X_test_meta,  y_test  = build_Xy_binary(df[df['split']=='test'])

print('Shapes:')
print('X_train_img', X_train_img.shape, 'X_train_meta', X_train_meta.shape, 'y_train', y_train.shape)
print('X_val_img', X_val_img.shape, 'X_val_meta', X_val_meta.shape, 'y_val', y_val.shape)
print('X_test_img', X_test_img.shape, 'X_test_meta', X_test_meta.shape, 'y_test', y_test.shape)

Using image column: image_id


100%|██████████| 5859/5859 [02:04<00:00, 46.88it/s] 


Shapes:
X_train_img (46860, 100, 100, 3) X_train_meta (46860, 1) y_train (46860,)
X_val_img (5859, 100, 100, 3) X_val_meta (5859, 1) y_val (5859,)
X_test_img (5859, 100, 100, 3) X_test_meta (5859, 1) y_test (5859,)


4. ANN Baseline (Flattened + isNight)

In [5]:

X_train_flat = X_train_img.reshape((X_train_img.shape[0], -1))
X_val_flat   = X_val_img.reshape((X_val_img.shape[0], -1))
X_test_flat  = X_test_img.reshape((X_test_img.shape[0], -1))

X_train_flat = np.hstack([X_train_flat, X_train_meta])
X_val_flat   = np.hstack([X_val_flat, X_val_meta])
X_test_flat  = np.hstack([X_test_flat, X_test_meta])

ann_model = Sequential([
    Input(shape=(X_train_flat.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │     3,840,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,848,577 (14.68 MB)

 Trainable params: 3,848,577 (14.68 MB)

 Non-trainable params: 0 (0.00 B)

5. Train ANN Baseline (Weighted, Early Stop, Checkpoint)

In [6]:
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True, 
    verbose=1
)

checkpoint = ModelCheckpoint(
    'fruit_quality_ann_detector.keras', 
    monitor='val_loss', 
    save_best_only=True, 
    verbose=1
)

cw = class_weight.compute_class_weight(
    'balanced', 
    classes=np.unique(y_train), 
    y=y_train
)

cw_dict = {int(i): float(w) for i,w in enumerate(cw)}
print('Class weights:', cw_dict)

history_ann = ann_model.fit(
    X_train_flat, y_train, 
    validation_data=(X_val_flat, y_val), 
    epochs=NUM_EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=[early_stop, checkpoint], 
    class_weight=cw_dict, 
    shuffle=True
)

Class weights: {0: 1.5, 1: 0.75}
Epoch 1/30
733/733 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5497 - loss: 1.0128 - precision: 0.7098 - recall: 0.5469
Epoch 1: val_loss improved from None to 0.55109, saving model to fruit_quality_ann_detector.keras
733/733 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.5865 - loss: 0.7583 - precision: 0.7408 - recall: 0.5841 - val_accuracy: 0.6214 - val_loss: 0.5511 - val_precision: 0.6755 - val_recall: 0.8318
Epoch 2/30
733/733 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7199 - loss: 0.5339 - precision: 0.8596 - recall: 0.6923
Epoch 2: val_loss improved from 0.55109 to 0.37628, saving model to fruit_quality_ann_detector.keras
733/733 ━━━━━━━━━━━━━━━━━━━━ 21s 29ms/step - accuracy: 0.7761 - loss: 0.4491 - precision: 0.9034 - recall: 0.7438 - val_accuracy: 0.8459 - val_loss: 0.3763 - val_precision: 0.9248 - val_recall: 0.8369
Epoch 3/30
733/733 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.8670 - loss: 0.2932 - precision: 0.9571 - recall: 0

6. CNN Binary Model with Meta Input (isNight)

In [ ]:
""" from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout, Input, concatenate
from tensorflow.keras.models import Model
import tensorflow as tf

def build_cnn_binary(input_shape=(IMG_SIZE, IMG_SIZE, NUM_CHANNELS)):
    # ---- ΕΙΣΟΔΟΣ ΕΙΚΟΝΑΣ ----
    img_in = Input(shape=input_shape, name='image')

    # ---- ΣΥΝΑΡΜΟΛΟΓΗΣΗ CNN ----
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(img_in)
    x = MaxPool2D(pool_size=(2, 2))(x)
    x = Dropout(0.2)(x)

    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPool2D(pool_size=(2, 2))(x)
    x = Dropout(0.3)(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPool2D(pool_size=(2, 2))(x)
    x = Dropout(0.4)(x)

    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    # ---- ΕΙΣΟΔΟΣ META FEATURE ----
    meta_in = Input(shape=(1,), name='isNight')
    m = Dense(16, activation='relu')(meta_in)

    # ---- ΣΥΝΔΥΑΣΜΟΣ ----
    combined = concatenate([x, m])
    combined = Dense(64, activation='relu')(combined)
    combined = Dropout(0.3)(combined)

    out = Dense(1, activation='sigmoid')(combined)

    # ---- ΟΡΙΣΜΟΣ ΚΑΙ ΣΥΝΘΕΣΗ ----
    model = Model(inputs=[img_in, meta_in], outputs=out, name='cnn_binary')
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model
 """

In [7]:
def build_cnn_binary(input_shape=(IMG_SIZE, IMG_SIZE, NUM_CHANNELS)):
    img_in = Input(shape=input_shape, name='image')
    x = Conv2D(32, 3, activation='relu', padding='same')(img_in)
    x = MaxPool2D()(x)
    x = Conv2D(64, 3, activation='relu', padding='same')(x)
    x = MaxPool2D()(x)
    x = Conv2D(128, 3, activation='relu', padding='same')(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(64, activation='relu')(x)

    meta_in = Input(shape=(1,), name='isNight')
    m = Dense(8, activation='relu')(meta_in)

    combined = concatenate([x, m])
    combined = Dropout(0.3)(combined)

    out = Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[img_in, meta_in], outputs=out, name='cnn_binary')
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
    )
    return model

cnn_model = build_cnn_binary(input_shape=(IMG_SIZE, IMG_SIZE, NUM_CHANNELS))
cnn_model.summary()

Model: "cnn_binary"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 100, 100,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 100, 100,  │        896 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 50, 50,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 50, 50,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 25, 25,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 25, 25,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ isNight             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 8)         │         16 │ isNight[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 72)        │          0 │ dense_3[0][0],    │
│ (Concatenate)       │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 72)        │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         73 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 101,593 (396.85 KB)

 Trainable params: 101,593 (396.85 KB)

 Non-trainable params: 0 (0.00 B)

7. Train CNN Detector (Image + isNight)

In [ ]:
""" early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint_cnn = ModelCheckpoint(
    'fruit_quality_cnn_detector.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

cnn_model = build_cnn_binary()
cnn_model.summary()

cw = class_weight.compute_class_weight(
    'balanced', 
    classes=np.unique(y_train), 
    y=y_train
)

cw_dict = {int(i): float(w) for i,w in enumerate(cw)}
print('Class weights:', cw_dict)

history_cnn = cnn_model.fit(
    [X_train_img, X_train_meta], y_train,
    validation_data=([X_val_img, X_val_meta], y_val),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr, checkpoint_cnn],
    class_weight=cw_dict,    # <— βγάλ’ το αν δεν έτρεξες το Κελί 4
    shuffle=True,
    verbose=1
)
 """

Model: "cnn_binary"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 50, 50, 3) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 50, 50,    │        896 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 25, 25,    │          0 │ conv2d_9[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 25, 25,    │          0 │ max_pooling2d_9[… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 25, 25,    │     18,496 │ dropout_15[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 12, 12,    │          0 │ conv2d_10[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 12, 12,    │          0 │ max_pooling2d_10… │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 12, 12,    │     73,856 │ dropout_16[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, 6, 6, 128) │          0 │ conv2d_11[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 6, 6, 128) │          0 │ max_pooling2d_11… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 4608)      │          0 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │    589,952 │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ isNight             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 128)       │          0 │ dense_12[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 16)        │         32 │ isNight[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 144)       │          0 │ dropout_18[0][0], │
│ (Concatenate)       │                   │            │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 64)        │      9,280 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 64)        │          0 │ dense_14[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 692,577 (2.64 MB)

 Trainable params: 692,577 (2.64 MB)

 Non-trainable params: 0 (0.00 B)

Class weights: {0: 1.5, 1: 0.75}
Epoch 1/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6866 - loss: 0.5164 - precision: 0.7925 - recall: 0.7105
Epoch 1: val_loss improved from None to 0.05689, saving model to fruit_quality_cnn_detector.keras
733/733 ━━━━━━━━━━━━━━━━━━━━ 34s 44ms/step - accuracy: 0.8386 - loss: 0.3093 - precision: 0.9007 - recall: 0.8517 - val_accuracy: 0.9822 - val_loss: 0.0569 - val_precision: 0.9963 - val_recall: 0.9770 - learning_rate: 0.0010
Epoch 2/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.9400 - loss: 0.1667 - precision: 0.9715 - recall: 0.9372
Epoch 2: val_loss improved from 0.05689 to 0.03120, saving model to fruit_quality_cnn_detector.keras
733/733 ━━━━━━━━━━━━━━━━━━━━ 40s 55ms/step - accuracy: 0.9583 - loss: 0.1196 - precision: 0.9806 - recall: 0.9564 - val_accuracy: 0.9899 - val_loss: 0.0312 - val_precision: 0.9933 - val_recall: 0.9916 - learning_rate: 0.0010
Epoch 3/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.979

In [8]:
checkpoint_cnn = ModelCheckpoint(
    'fruit_quality_cnn_detector.keras', 
    monitor='val_loss', 
    save_best_only=True, 
    verbose=1
)

history_cnn = cnn_model.fit(
    [X_train_img, X_train_meta], y_train, 
    validation_data=([X_val_img, X_val_meta], y_val), 
    epochs=NUM_EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=[early_stop, checkpoint_cnn], 
    class_weight=cw_dict, 
    shuffle=True
)

Epoch 1/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.7722 - loss: 0.4147 - precision: 0.8575 - recall: 0.7745
Epoch 1: val_loss improved from None to 0.05060, saving model to fruit_quality_cnn_detector.keras
733/733 ━━━━━━━━━━━━━━━━━━━━ 110s 147ms/step - accuracy: 0.8981 - loss: 0.2276 - precision: 0.9388 - recall: 0.9063 - val_accuracy: 0.9838 - val_loss: 0.0506 - val_precision: 0.9823 - val_recall: 0.9936
Epoch 2/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.9794 - loss: 0.0611 - precision: 0.9888 - recall: 0.9802
Epoch 2: val_loss did not improve from 0.05060
733/733 ━━━━━━━━━━━━━━━━━━━━ 107s 146ms/step - accuracy: 0.9829 - loss: 0.0519 - precision: 0.9912 - recall: 0.9832 - val_accuracy: 0.9836 - val_loss: 0.0513 - val_precision: 0.9837 - val_recall: 0.9918
Epoch 3/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.9571 - loss: 0.1113 - precision: 0.9801 - recall: 0.9547
Epoch 3: val_loss improved from 0.05060 to 0.02534, saving model to fruit_

8. RNN (BiLSTM) Binary Detector (Image → Sequence + isNight)

In [9]:
def image_to_sequence(imgs):
    N = imgs.shape[0]
    return imgs.reshape((N, IMG_SIZE, IMG_SIZE * NUM_CHANNELS))

def build_rnn_binary(time_steps=IMG_SIZE, features=IMG_SIZE*NUM_CHANNELS):
    seq_in = Input(shape=(time_steps, features), name='sequence')
    x = Bidirectional(LSTM(64))(seq_in)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)

    meta_in = Input(shape=(1,), name='isNight')
    m = Dense(8, activation='relu')(meta_in)

    combined = concatenate([x, m])
    out = Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[seq_in, meta_in], outputs=out, name='rnn_binary')
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
    return model

X_train_seq = image_to_sequence(X_train_img)
X_val_seq   = image_to_sequence(X_val_img)
X_test_seq  = image_to_sequence(X_test_img)

rnn_model = build_rnn_binary()
rnn_model.summary()

Model: "rnn_binary"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 100, 300)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │    186,880 │ sequence[0][0]    │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ isNight             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 8)         │         16 │ isNight[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 72)        │          0 │ dense_6[0][0],    │
│ (Concatenate)       │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │         73 │ concatenate_1[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 195,225 (762.60 KB)

 Trainable params: 195,225 (762.60 KB)

 Non-trainable params: 0 (0.00 B)

9. Train RNN Binary Detector

In [10]:
checkpoint_rnn = ModelCheckpoint(
    'fruit_quality_rnn_detector.keras', 
    monitor='val_loss', 
    save_best_only=True, verbose=1
)

history_rnn = rnn_model.fit(
    [X_train_seq, X_train_meta], y_train, 
    validation_data=([X_val_seq, X_val_meta], y_val), 
    epochs=NUM_EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=[early_stop, checkpoint_rnn], 
    class_weight=cw_dict, 
    shuffle=True
)

Epoch 1/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5349 - loss: 0.6866 - precision: 0.7050 - recall: 0.5170
Epoch 1: val_loss improved from None to 0.59829, saving model to fruit_quality_rnn_detector.keras
733/733 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.5489 - loss: 0.6676 - precision: 0.7355 - recall: 0.5049 - val_accuracy: 0.5972 - val_loss: 0.5983 - val_precision: 0.8838 - val_recall: 0.4557
Epoch 2/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.6432 - loss: 0.5748 - precision: 0.8577 - recall: 0.5552
Epoch 2: val_loss improved from 0.59829 to 0.39301, saving model to fruit_quality_rnn_detector.keras
733/733 ━━━━━━━━━━━━━━━━━━━━ 39s 53ms/step - accuracy: 0.7009 - loss: 0.5225 - precision: 0.8864 - recall: 0.6325 - val_accuracy: 0.8128 - val_loss: 0.3930 - val_precision: 0.9328 - val_recall: 0.7750
Epoch 3/30
732/733 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8205 - loss: 0.3627 - precision: 0.9320 - recall: 0.7878
Epoch 3: val_loss improved 

10. Evaluation (Reports + Confusion Matrices)

In [11]:
BATCH_EVAL = 512 

def _to_labels(y):
    y = np.asarray(y)
    return np.argmax(y, axis=1) if y.ndim == 2 else y.ravel()

y_true = _to_labels(y_test)

if os.path.exists('fruit_quality_ann_detector.keras'):
    ann_model.load_weights('fruit_quality_ann_detector.keras')
if os.path.exists('fruit_quality_cnn_detector.keras'):
    cnn_model.load_weights('fruit_quality_cnn_detector.keras')

rnn_ckpt = 'fruit_quality_rnn_detector.keras'
if os.path.exists(rnn_ckpt):
    rnn_model.load_weights(rnn_ckpt)

try:
    y_pred = ann_model.predict(X_test_flat, batch_size=BATCH_EVAL)
    y_pred_ann = (y_pred.ravel() > 0.5).astype(int) if y_pred.ndim == 2 and y_pred.shape[1] == 1 else np.argmax(y_pred, axis=1)
    print("=== ANN ===")
    print(classification_report(y_true, y_pred_ann, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_ann))
except Exception as e:
    print("ANN eval failed:", e)

try:
    y_pred = cnn_model.predict([X_test_img, X_test_meta], batch_size=BATCH_EVAL)
    y_pred_cnn = (y_pred.ravel() > 0.5).astype(int) if y_pred.ndim == 2 and y_pred.shape[1] == 1 else np.argmax(y_pred, axis=1)
    print("\n=== CNN ===")
    print(classification_report(y_true, y_pred_cnn, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_cnn))
except Exception as e:
    print("CNN eval failed:", e)

try:
    y_pred = rnn_model.predict([X_test_seq, X_test_meta], batch_size=BATCH_EVAL)
    y_pred_rnn = (y_pred.ravel() > 0.5).astype(int) if y_pred.ndim == 2 and y_pred.shape[1] == 1 else np.argmax(y_pred, axis=1)
    print("\n=== RNN ===")
    print(classification_report(y_true, y_pred_rnn, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_rnn))
except Exception as e:
    print("RNN eval failed:", e)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
=== ANN ===
              precision    recall  f1-score   support

           0     0.9141    0.9805    0.9461      1953
           1     0.9899    0.9539    0.9716      3906

    accuracy                         0.9628      5859
   macro avg     0.9520    0.9672    0.9589      5859
weighted avg     0.9646    0.9628    0.9631      5859

Confusion matrix:
 [[1915   38]
 [ 180 3726]]
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 301ms/step

=== CNN ===
              precision    recall  f1-score   support

           0     0.9909    0.9985    0.9946      1953
           1     0.9992    0.9954    0.9973      3906

    accuracy                         0.9964      5859
   macro avg     0.9950    0.9969    0.9960      5859
weighted avg     0.9964    0.9964    0.9964      5859

Confusion matrix:
 [[1950    3]
 [  18 3888]]
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 127ms/step

=== RNN ===
              precision    recall  f1-score   support

           0     0.8571    0.9831    0.

## SemiSupervised Lifelong CNN Adversarial Detector